# NORT
The following is an example of an implmentation of NORT analysis with the use of the bikipy package!
Please note that the package is still in early alpha, and many things might change, the notebook will be updated accordingly,
so you can check back any time.

In [ ]:
from pathlib import Path
import pickle
import glob
import re
import os

import pandas as pd
import numpy as np

from bikipy.behaviour.nort.experiment import NortExperiment
from bikipy.behaviour.nort.trial import NortField

## Define project
We start with defining the root of the project. This should be the directory where you store all your project related files, and documents. The analysis results will be saved here also.

In [ ]:
PROJECT_DIRECTORY = Path("Paste your path here!")

## Annotation of objects
Any post-habituation, training and novelty, session with NORT includes two objects, and we need to annotate both of them. We also need to note the variable, the constnat, and the novel objects. This is automated within the NortField class, 

These annotations will be saved into a NortField object for use in analysis



In [ ]:
COCO_FILE_PATH = PROJECT_DIRECTORY / "add" / "relative" / "path" / "here"
TRAINING_IMAGE_PATH = PROJECT_DIRECTORY / "add" / "relative" / "path" / "here"   # optional

perimeters = PolygonalPerimeter.from_coco(COCO_FILE_PATH)
nort_field = NortField(
    constant_object_perimeter=perimeters["constant"],
    variable_object_perimeter=PolygonalPerimeter["variable"],
    novel_object_perimeter=PolygonalPerimeter["novel"],
    inspect_image=TRAINING_IMAGE_PATH
)

## Prepare data
Now that we have the `NortField` objects prepared, we can prepare the data that will be analysed. Our dataset is from DeepLabCut, which is why we will use the `reader.DeepLabCutReader` to import the data to the pipeline.

Note that we have converted our DeepLabCut data to parquet, for a better long-term solution, hdf and csv is fully supported if you use those, native, formats.

### Define directories

In [ ]:
DEEPLABCUT_DIR = Path("/mnt/md0/Projects/Neuroscience/Imen/data/nort")

EXPERIMENT_DIR = DEEPLABCUT_DIR / "Experiment_1"

WORKING_DIR = Path(".").resolve()
DATA_DIR = WORKING_DIR / "data"
IMAGE_DIR = DATA_DIR / "area_images"

We also store the trial ID as the only integer in the filename, which is why we can use a simple regex parser to extract the experiment ID from the filename

In [ ]:
EXP_ID_REGEX_PATTERN = re.compile("\d+")

### Collect data in one dictionary

In [ ]:
trial_id_range_vs_exp_meta = {}

for video_path, data_path in zip(
    glob(str(experiment_dir / "**" / "*.mp4")),
    glob(str(experiment_dir / "**" / "*.parquet")),
):
    # Get the trial_id from the filename of the video
    trial_id = int(EXP_ID_REGEX_PATTERN.findall(Path(video_path).stem)[0])

    _, width, height, fps = get_video_data(video_path)
    trial_id_range_vs_exp_meta[trial_id] = {
        # The path to the coordinate data; hdf, csv, etc...
        "coordinate_data_path": data_path,
        
        # Path to video will be used to define FPS and resolution
        "video_path": video_path,

        # # In the case that you have more than one NortField, you can designate the field to the trial ID.
        # # Don't use this kwarg if you only have one field. The trial_id_vs_stage dict needs to be defined, however.
        # "stage": (stage := trial_id_vs_stage[trial_id]),
        
        # Since we only have one field
        "stage": nort_field,

        # # Optional. The ID of the animal in the respective trial. The animal ID is used to group
        # # habituation, traning, and novelty events of the animal.The trial_id_vs_animal_id dict needs to be defined, however.
        # "animal_id": (animal_id := trial_id_vs_animal_id[trial_id]),

        # Inspect the filters. Usefull for debugging the analysis.
        "inspect": False,
    }

## Run the analysis

In [ ]:
df = NortExperiment(
    trial_id_vs_data=trial_id_range_vs_exp_meta,
    metric_resolution=0.4,
    nose_label="nose",

    # The eye is between the left and the right ear
    eye_center_label="mid-left_ear-right_ear",

    # The torso is between the eye center and tail
    torso_label="mid-mid-left_ear-right_ear-tail",

    nort_field_vs_nort_field_object=nort_field_vs_nort_field_object,
    perimeter_border_normal_metric_magnitude=0.04,
    center_metric_length=0.2,
    maximum_radians_inter_gaze_perimeter=0.25 * np.pi,
    init_from="parquet",

    midpoint_groups=[
        ("left_ear", "right_ear"),              # -> eye_center
        ("mid-left_ear-right_ear", "tail"),     # -> torso
    ],
).df